# 03 - Polyvore / Outfit Compatibility V0 - Audit dataset

Objectif : inspecter un dataset Polyvore/outfit, produire un rapport d'audit, et s'arreter avant tout entrainement.

Le module doit rester aligne avec Fashion V1.1 : `product_type_v0`, `canonical_category`, `outfit_role`.


## 1. Monter Google Drive


In [ ]:
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path('/content/drive')
if (DRIVE_MOUNT / 'MyDrive').exists():
    print('Google Drive deja monte.')
else:
    drive.mount(str(DRIVE_MOUNT))

DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive'
print(f'Drive root: {DRIVE_ROOT}')


## 2. Cloner ou mettre a jour le repo


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/MilFhey/fit-outfit-advisor.git'
BRANCH = 'main'
REPO_DIR = Path('/content/fit-outfit-advisor-repo')
PROJECT_DIR = REPO_DIR / 'fit-outfit-advisor'

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print('Repo existant : mise a jour.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Dossier projet absent : {PROJECT_DIR}')

sys.path.insert(0, str(PROJECT_DIR))
print(f'Projet pret : {PROJECT_DIR}')


## 3. Installer les dependances


In [ ]:
requirements_path = PROJECT_DIR / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError(f'Requirements absent : {requirements_path}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'kaggle'], check=True)


## 4. Creer les dossiers temporaires


In [ ]:
RUNTIME_ROOT = Path('/content/fit-outfit-runtime')
POLYVORE_ROOT = RUNTIME_ROOT / 'polyvore'
DOWNLOAD_DIR = POLYVORE_ROOT / 'downloads'
EXTRACT_DIR = POLYVORE_ROOT / 'extracted'
REPORT_DIR = PROJECT_DIR / 'reports'
for directory in [RUNTIME_ROOT, POLYVORE_ROOT, DOWNLOAD_DIR, EXTRACT_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print(f'Runtime root: {RUNTIME_ROOT}')


## 5. Configurer la source dataset

Renseigne soit `KAGGLE_DATASET_SLUG`, soit `DRIVE_ZIP_PATH`. Laisse vide ce que tu n'utilises pas.


In [ ]:
KAGGLE_DATASET_SLUG = ''  # exemple: 'username/polyvore-dataset'
DRIVE_ZIP_PATH = ''       # exemple: '/content/drive/MyDrive/datasets/polyvore.zip'

if not KAGGLE_DATASET_SLUG and not DRIVE_ZIP_PATH:
    raise ValueError('Renseigne KAGGLE_DATASET_SLUG ou DRIVE_ZIP_PATH avant de continuer.')


## 6. Telecharger ou extraire le dataset


In [ ]:
import json
import zipfile

if KAGGLE_DATASET_SLUG:
    from google.colab import userdata

    kaggle_token = userdata.get('KAGGLE_API')
    if not kaggle_token:
        raise ValueError('Secret Colab KAGGLE_API absent.')
    os.environ['KAGGLE_KEY'] = kaggle_token
    # Pour les tokens KGAT, Kaggle CLI lit KAGGLE_KEY ; KAGGLE_USERNAME peut rester une valeur factice.
    os.environ.setdefault('KAGGLE_USERNAME', 'colab')
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET_SLUG, '-p', str(DOWNLOAD_DIR), '--unzip'],
        check=True,
    )
    dataset_root = DOWNLOAD_DIR
else:
    source_zip = Path(DRIVE_ZIP_PATH)
    if not source_zip.exists():
        raise FileNotFoundError(f'Zip Drive absent : {source_zip}')
    with zipfile.ZipFile(source_zip, 'r') as archive:
        archive.extractall(EXTRACT_DIR)
    dataset_root = EXTRACT_DIR

print(f'Dataset root: {dataset_root}')
print('Exemples de fichiers:')
for path in list(dataset_root.rglob('*'))[:30]:
    print(path.relative_to(dataset_root))


## 7. Detecter fichiers et structures candidates


In [ ]:
import pandas as pd

all_files = [path for path in dataset_root.rglob('*') if path.is_file()]
tabular_files = [path for path in all_files if path.suffix.lower() in {'.csv', '.json', '.jsonl'}]
image_files = [path for path in all_files if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]

def classify_file(path: Path) -> str:
    name = path.name.lower()
    if 'outfit' in name or 'set' in name:
        return 'outfit_candidate'
    if 'item' in name or 'product' in name:
        return 'item_candidate'
    if 'category' in name or 'metadata' in name or 'meta' in name:
        return 'metadata_candidate'
    return 'other_tabular'

file_summary = []
for path in tabular_files:
    file_summary.append({
        'relative_path': str(path.relative_to(dataset_root)),
        'suffix': path.suffix.lower(),
        'kind_guess': classify_file(path),
        'size_bytes': path.stat().st_size,
    })

file_summary_df = pd.DataFrame(file_summary).sort_values(['kind_guess', 'relative_path'])
display(file_summary_df.head(80))
print(f'Tabular files: {len(tabular_files)}')
print(f'Image files: {len(image_files)}')


## 8. Inspecter les fichiers tabulaires lisibles


In [ ]:
def read_table_sample(path: Path):
    try:
        if path.suffix.lower() == '.csv':
            return pd.read_csv(path, nrows=5000, low_memory=False)
        if path.suffix.lower() == '.jsonl':
            return pd.read_json(path, lines=True, nrows=5000)
        if path.suffix.lower() == '.json':
            try:
                data = json.loads(path.read_text(encoding='utf-8'))
            except UnicodeDecodeError:
                data = json.loads(path.read_text(encoding='utf-8-sig'))
            if isinstance(data, list):
                return pd.json_normalize(data[:5000])
            if isinstance(data, dict):
                for value in data.values():
                    if isinstance(value, list):
                        return pd.json_normalize(value[:5000])
                return pd.json_normalize(data)
    except Exception as exc:
        return exc
    return ValueError(f'Format non gere: {path}')

table_reports = []
for path in tabular_files[:80]:
    sample = read_table_sample(path)
    if isinstance(sample, Exception):
        table_reports.append({
            'relative_path': str(path.relative_to(dataset_root)),
            'readable': False,
            'error': repr(sample),
        })
        continue
    columns = list(sample.columns)
    table_reports.append({
        'relative_path': str(path.relative_to(dataset_root)),
        'readable': True,
        'sample_shape': list(sample.shape),
        'columns': columns,
        'missing_pct_top': sample.isna().mean().sort_values(ascending=False).head(10).round(4).to_dict(),
        'id_like_columns': [col for col in columns if 'id' in str(col).lower()],
        'category_like_columns': [col for col in columns if any(key in str(col).lower() for key in ['cat', 'type', 'label', 'name'])],
        'outfit_like_columns': [col for col in columns if any(key in str(col).lower() for key in ['outfit', 'set'])],
    })

table_reports_df = pd.DataFrame(table_reports)
display(table_reports_df[['relative_path', 'readable', 'sample_shape', 'id_like_columns', 'category_like_columns', 'outfit_like_columns']].head(80))


## 9. Inspecter la configuration Outfit V1 brouillon


In [ ]:
from src.mappings.polyvore_mapping import load_outfit_v1_config, validate_outfit_v1_config

OUTFIT_CONFIG_PATH = PROJECT_DIR / 'config' / 'outfit_v1_config.json'
outfit_config = load_outfit_v1_config(OUTFIT_CONFIG_PATH)
validate_outfit_v1_config(outfit_config)
print(json.dumps(outfit_config, indent=2, ensure_ascii=False))
if outfit_config.get('status') == 'draft_requires_dataset_inspection':
    print('Config brouillon : aucun entrainement autorise.')


## 10. Generer le rapport d'audit


In [ ]:
report = {
    'version': 'polyvore_v0_dataset_audit',
    'dataset_root': str(dataset_root),
    'source': {
        'kaggle_dataset_slug': KAGGLE_DATASET_SLUG,
        'drive_zip_path': DRIVE_ZIP_PATH,
    },
    'file_counts': {
        'total_files': len(all_files),
        'tabular_files': len(tabular_files),
        'image_files': len(image_files),
    },
    'tabular_file_summary': file_summary,
    'table_reports': table_reports,
    'outfit_config_status': outfit_config.get('status'),
    'taxonomy_alignment_required': ['product_type_v0', 'canonical_category', 'outfit_role'],
    'training_executed': False,
    'next_decision': 'Completer polyvore_label_mapping apres inspection humaine des labels reels.',
}

REPORT_PATH = REPORT_DIR / 'polyvore_v0_dataset_audit.json'
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Rapport ecrit : {REPORT_PATH}')

drive_report_dir = DRIVE_ROOT / 'fit-outfit-advisor' / 'reports'
drive_report_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPORT_PATH, drive_report_dir / REPORT_PATH.name)
print(f'Rapport copie Drive : {drive_report_dir / REPORT_PATH.name}')


## 11. Arret volontaire avant entrainement


In [ ]:
raise SystemExit(
    'Audit Polyvore termine. Aucun entrainement lance. '
    'Envoie reports/polyvore_v0_dataset_audit.json a Codex pour valider le mapping.'
)
